# Train a small model: everyday English → Structured English

Part of **natural-to-binary**. A small, free model learns to rewrite everyday requests ("a timer for my studying") into Structured English, which our rule compiler turns into a WebAssembly app. The model never writes bytes; the compiler does that part, and it never guesses.

**You need:** a free Google Colab GPU (*Runtime → Change runtime type → T4 GPU*) and a `pairs.jsonl` file from the lab page (*Training set builder → Copy as JSONL*, saved as a file).

**Score:** the share of held-out requests whose output **compiles**, before and after training. That's the same score the lab gives Claude, so the two can be compared directly.

*Written alongside the project but not yet run on a GPU. Library versions may need adjusting.*

In [ ]:
!pip -q install "transformers>=4.44" "peft>=0.12" "trl>=0.11" "datasets>=2.20" accelerate
!node --version || (apt-get -qq install -y nodejs)
!git clone -q https://github.com/AbuSayeed917/natural-to-binary.git

## 1. Load the training pairs
Each line of `pairs.jsonl` is `{"everyday": ..., "structured": ...}`.

In [ ]:
import json, random, os
if not os.path.exists('pairs.jsonl'):
    from google.colab import files
    files.upload()   # choose pairs.jsonl
pairs = [json.loads(l) for l in open('pairs.jsonl') if l.strip()]
random.seed(0)
# Hold out whole apps, not just sentences, so the model is tested on apps it never trained on.
programs = sorted({p['structured'] for p in pairs}); random.shuffle(programs)
test_programs = set(programs[: max(1, len(programs) // 5)])
train = [p for p in pairs if p['structured'] not in test_programs]
test = [p for p in pairs if p['structured'] in test_programs]
print(len(pairs), 'pairs |', len(train), 'train |', len(test), 'test, on', len(test_programs), 'held-out apps')

## 2. Ask our compiler whether a program compiles

In [ ]:
import subprocess
def compiles(programs):
    lines = '\n'.join(json.dumps({'id': i, 'program': p}) for i, p in enumerate(programs))
    out = subprocess.run(['node', 'natural-to-binary/kit/check.js', 'natural-to-binary/compilers/structured-english-v3'],
                         input=lines, capture_output=True, text=True).stdout
    return [json.loads(l)['ok'] for l in out.splitlines()]
print('sanity check:', compiles([train[0]['structured'], 'Every fortnight:\n  Clear x.']))   # expect [True, False]

## 3. Load a small open model
`Qwen2.5-1.5B-Instruct` already understands English and fits a free T4 GPU.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map='auto')
SYSTEM = 'Rewrite the request as a Structured English v0.3 program. Reply with the program only.'
def prompt(request):
    return tok.apply_chat_template([{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': request}], tokenize=False, add_generation_prompt=True)
def generate(requests, max_new_tokens=700):
    outs = []
    for r in requests:
        ids = tok(prompt(r), return_tensors='pt').to(model.device)
        with torch.no_grad():
            g = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
        outs.append(tok.decode(g[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip())
    return outs

## 4. Score before training (baseline)

In [ ]:
test_requests = [p['everyday'] for p in test][:40]
before = generate(test_requests)
ok_before = compiles(before)
print(f'BEFORE training: {sum(ok_before)}/{len(ok_before)} compile ({100*sum(ok_before)/len(ok_before):.0f}%)')

## 5. Train with LoRA (a small add-on, so it fits a free GPU)

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
ds = Dataset.from_list([{'messages': [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': p['everyday']}, {'role': 'assistant', 'content': p['structured']}]} for p in train])
trainer = SFTTrainer(
    model=model, train_dataset=ds,
    peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules='all-linear', task_type='CAUSAL_LM'),
    args=SFTConfig(output_dir='se-lora', num_train_epochs=3, per_device_train_batch_size=2, gradient_accumulation_steps=4,
                   learning_rate=2e-4, logging_steps=10, save_strategy='no', fp16=True, max_seq_length=1024, report_to=[]),
)
trainer.train()
model = trainer.model

## 6. Score after training

In [ ]:
after = generate(test_requests)
ok_after = compiles(after)
print(f'BEFORE: {sum(ok_before)}/{len(ok_before)} compile   AFTER: {sum(ok_after)}/{len(ok_after)} compile')
for r, o, ok in list(zip(test_requests, after, ok_after))[:3]:
    print('\n---', 'OK' if ok else 'FAILED', '---', r, '\n', o[:400])

## 7. Save the adapter (a few megabytes)

In [ ]:
trainer.model.save_pretrained('se-lora-adapter')
!zip -qr se-lora-adapter.zip se-lora-adapter
from google.colab import files
files.download('se-lora-adapter.zip')